In [ ]:
# All imports and configuration. Install dependencies outside the notebook.
import copy,csv,datetime as dt,gc,hashlib,json,math,os,platform,shutil,subprocess,sys,time,uuid
from pathlib import Path
from collections import defaultdict
import numpy as np
from PIL import Image
from skimage.metrics import structural_similarity
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
import lpips
try:
    from diffusers import StableDiffusionInpaintPipeline, DDIMScheduler
except ImportError:
    StableDiffusionInpaintPipeline=DDIMScheduler=None
ID='18-spatial-attack-transfer'
IN_COLAB=Path('/content').is_dir() and sys.platform=='linux'
ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/ID/'experiment.ipynb').is_file()),None)
assert ROOT is not None, 'Run the notebook from the repository root or its src directory'
EXP=ROOT/'src'/ID; DATA=ROOT/'data/unified17-coco'
os.environ['TORCH_HOME']=str(EXP/'runs/perceptual-cache')
PROFILE=os.environ.get('TRANSFER_PROFILE','smoke');assert PROFILE in ('smoke','benchmark','cheap','lama','sd15','sd2','report')
if not IN_COLAB:assert Path(sys.prefix).resolve()==(EXP/'.venv').resolve(),'Use own experiment .venv'
RUN_ID=os.environ.get('TRANSFER_RUN_ID','unified18-seed17-v1' if PROFILE not in ('smoke','benchmark') else PROFILE+'-'+uuid.uuid4().hex[:8])
assert Path(RUN_ID).name==RUN_ID
RUN=EXP/'runs'/RUN_ID
TRAIN_RUN=ROOT/'src/17-spatial-attack-training/runs'/os.environ.get('UNIFIED_TRAIN_RUN','unified17-seed17-v1')
SIZE,BITS,CHANNELS,AMPLITUDE=128,16,64,.03
SEED=17; AREAS=[819,1638,2458]; SIGMAS=[.02,.05,.1];EVAL_BATCH=64
EDITOR_BATCH=int(os.environ.get('TRANSFER_EDITOR_BATCH','16'));LAMA_BATCH=32
BENCH_BATCHES=[int(x) for x in os.environ.get('TRANSFER_BENCH_BATCHES','4,8,16,32').split(',')]
BENCH_EDITORS=os.environ.get('TRANSFER_BENCH_EDITORS','lama,sd15,sd2').split(',')
EDITORS={'sd15':dict(model_id='stable-diffusion-v1-5/stable-diffusion-inpainting',revision='8a4288a76071f7280aedbdb3253bdb9e9d5d84bb'),'sd2':dict(model_id='sd2-community/stable-diffusion-2-inpainting',revision='5f74973cbb64c8568780732c17f43eb269d63a0d')}
EDITOR_ROOT=ROOT/'editors'
LAMA_PATH=EDITOR_ROOT/'big-lama.pt'
LAMA_SHA='344c77bbcb158f17dd143070d1e789f38a66c04202311ae3a258ef66667a9ea9'
PROMPT='a patch of blue ceramic mosaic tiles, detailed photograph, natural lighting'
NEGATIVE='text, watermark, letters, blurry'
STEPS=20;GUIDANCE=7.5;STRENGTH=1.;EDITOR_SEED=181706
BOOT_SEED=181204;BOOT_N=10000
DEADLINE=time.time()+float(os.environ.get('TRANSFER_TOTAL_HOURS','12'))*3600
CUDA=torch.cuda.is_available();DEVICE=torch.device('cuda' if CUDA else 'cpu');AMP=CUDA
torch.set_num_threads(4);torch.backends.cudnn.benchmark=CUDA
CONFIG=dict(seed=SEED,size=SIZE,bits=BITS,amplitude=AMPLITUDE,areas=AREAS,sigmas=SIGMAS,editors=EDITORS,steps=STEPS,guidance=GUIDANCE,prompt=PROMPT,negative=NEGATIVE,editor_batch=EDITOR_BATCH,lama_sha=LAMA_SHA,bootstrap=BOOT_N,deadline=DEADLINE)


Описание: [проверка переноса устойчивости](../../README.md). Test доступен только после проверки frozen handoff всех 20 моделей. Benchmark и smoke используют validation.


In [ ]:
def sha(path):
    with Path(path).open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
def write(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.partial');tmp.write_text(json.dumps(value,indent=2,allow_nan=False)+'\n');os.replace(tmp,path)
def pack(path,**values):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.partial')
    with tmp.open('wb') as f:np.savez_compressed(f,**values)
    os.replace(tmp,path)
def csvwrite(path,rows):
    with Path(path).open('w',newline='') as f:w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def stamp(status,**kw):write(RUN/'progress.json',dict(status=status,time=dt.datetime.now(dt.timezone.utc).isoformat(),**kw))
def sync():
    if CUDA:torch.cuda.synchronize()
def deadline():
    if time.time()>DEADLINE:raise TimeoutError('Overall deadline reached; partial artifacts retained')
def fingerprint(v):return hashlib.sha256(json.dumps(v,sort_keys=True).encode()).hexdigest()
def rgb8(x):return (x.clamp(0,1)*255).round()/255.
def tensor(rgb):return torch.from_numpy(np.asarray(rgb).copy()).permute(0,3,1,2).to(DEVICE,dtype=torch.float32,memory_format=torch.channels_last)/255.
def to_rgb(x):return (x.detach().float().cpu().permute(0,2,3,1).numpy().clip(0,1)*255).round().astype(np.uint8)
def seed_for(i,r):return EDITOR_SEED+int(i)*997+r*71
RUN.mkdir(parents=True,exist_ok=True);SOURCE_SHA=sha(EXP/'experiment.ipynb')
if (RUN/'source.ipynb').exists():assert sha(RUN/'source.ipynb')==SOURCE_SHA,'Resume source changed'
else:shutil.copyfile(EXP/'experiment.ipynb',RUN/'source.ipynb')
write(RUN/'config.json',CONFIG);(RUN/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
write(RUN/'environment.json',dict(python=sys.version,torch=str(torch.__version__),gpu=torch.cuda.get_device_name() if CUDA else None,source=SOURCE_SHA))


In [ ]:
def normalization(ch):
 return nn.BatchNorm2d(ch)

class Block(nn.Sequential):
 def __init__(self,a,b):super().__init__(nn.Conv2d(a,b,3,padding=1,bias=False),normalization(b),nn.ReLU())

class CNN(nn.Module):
 def __init__(self,spatial=False):
  super().__init__();self.spatial=spatial;self.message_map=nn.Linear(BITS,BITS*16*16) if spatial else nn.Identity();self.features=nn.Sequential(*[Block(3 if i==0 else CHANNELS,CHANNELS) for i in range(4)])
  self.fuse=nn.Sequential(Block(CHANNELS+BITS+3,CHANNELS),nn.Conv2d(CHANNELS,3,1))
  self.decoder=nn.Sequential(*[Block(3 if i==0 else CHANNELS,CHANNELS) for i in range(7)],nn.AdaptiveAvgPool2d(4 if spatial else 1),nn.Flatten(),nn.Linear(CHANNELS*(16 if spatial else 1),128),nn.ReLU(),nn.Linear(128,BITS))
 def encode(self,x,bits):
  message=F.interpolate(self.message_map(bits*2-1).reshape(-1,BITS,16,16),size=(SIZE,SIZE),mode='nearest') if self.spatial else (bits*2-1)[:,:,None,None].expand(-1,-1,SIZE,SIZE)
  residual=AMPLITUDE*torch.tanh(self.fuse(torch.cat([self.features(x),message,x],1)))
  return (x+residual.float()).clamp(0,1)
 def decode(self,x):return self.decoder(x)

def autocast():return torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=AMP)

def errors(logits,bits):return (logits>=0)!=(bits>.5)

def psnr(mse):return -10*math.log10(max(float(mse),1e-12))

def batch_images(arrays,split,ids):return arrays[split][ids].to(DEVICE,dtype=torch.float32,non_blocking=CUDA,memory_format=torch.channels_last)/255.
def new_model():return CNN(spatial=True).to(device=DEVICE,memory_format=torch.channels_last)


In [ ]:
def load_bank():
    split='val' if PROFILE in ('smoke','benchmark') else 'test'
    pin=json.loads((DATA/'identity.json').read_text())
    for n in (f'{split}.npy',f'{split}_masks.npy',f'{split}_bits.npy','manifest.json'):assert sha(DATA/n)==pin['files'][n]
    cover=np.load(DATA/f'{split}.npy');masks=np.load(DATA/f'{split}_masks.npy',mmap_mode='r');bits=np.load(DATA/f'{split}_bits.npy')
    manifest=json.loads((DATA/'manifest.json').read_text());ids=np.array(manifest['splits'][split]['ids'])
    if PROFILE in ('smoke','benchmark'):
        ids=ids[:32];cover=cover[:32];masks=masks[:,:32];bits=bits[:32]
    assert len(set(ids.tolist()))==len(ids)
    return cover,masks,bits,ids

def load_handoff():
    path=TRAIN_RUN/'handoff.json';h=json.loads(path.read_text());assert len(h['arms'])==20 and h['seed']==17 and h['data_sha']==sha(DATA/'identity.json')
    for a,info in h['arms'].items():assert sha(TRAIN_RUN/info['path'])==info['sha256']
    return h

def mask_array(bank,idx,region,area=2):return np.unpackbits(bank[area,idx,region],axis=-1).reshape(-1,SIZE,SIZE).astype(bool)
def model_for(info):
    model=new_model();snap=torch.load(TRAIN_RUN/info['path'],map_location='cpu',weights_only=False);model.load_state_dict(snap['model']);model.eval();return model
@torch.inference_mode()
def decode(model,rgb):
    out=[]
    for lo in range(0,len(rgb),EVAL_BATCH):
        with autocast():z=model.decode(tensor(rgb[lo:lo+EVAL_BATCH])).float()
        assert torch.isfinite(z).all();out.append(z.cpu().numpy())
    return np.concatenate(out)
@torch.inference_mode()
def marking(model,info,cover,bits):
    out=[]
    for lo in range(0,len(cover),EVAL_BATCH):
        x=tensor(cover[lo:lo+EVAL_BATCH]);b=torch.from_numpy(bits[lo:lo+EVAL_BATCH]).to(DEVICE,dtype=torch.float32)
        with autocast():y=x+info['alpha']*(model.encode(x,b)-x)
        out.append(to_rgb(y))
    return np.concatenate(out)

def ensure_clean(arm,info,cover,bits,ids,identity):
    model=model_for(info);p=RUN/arm/'clean.npz'
    if p.exists():
        d=np.load(p);assert str(d['identity'])==identity and np.array_equal(d['ids'],ids) and np.array_equal(d['bits'],bits)
        marked=d['marked'];z=d['logits'];assert np.array_equal((decode(model,marked[:2])>=0),(z[:2]>=0))
    else:
        marked=marking(model,info,cover,bits);z=decode(model,marked)
        mse=((marked.astype(np.float32)-cover)/255.)**2;mse=mse.mean((1,2,3))
        ssim=np.array([structural_similarity(a,b,data_range=255,channel_axis=-1,win_size=11) for a,b in zip(cover,marked)])
        pack(p,identity=np.array(identity),ids=ids,bits=bits,marked=marked,logits=z,mse=mse,ssim=ssim)
    return model,marked,z

def save_condition(arm,name,ids,bits,clean,logits,identity,filtered=None,images=None,change=None,offset=0,energy=None):
    p=RUN/arm/name/f'{offset:04d}.npz'
    values=dict(ids=ids,bits=bits,clean=clean,logits=logits,identity=np.array(identity),filtered=np.zeros(len(ids),bool) if filtered is None else filtered)
    if energy is not None:values['attack_mse']=energy
    if images is not None:values['images']=images
    if change is not None:values['change_mae']=change
    pack(p,**values)

def cheap(cover,bank,bits,ids,h,identity):
    for arm,info in h['arms'].items():
        deadline();model,marked,clean=ensure_clean(arm,info,cover,bits,ids,identity)
        conditions={}
        noise=np.stack([np.random.default_rng(seed_for(i,7)).standard_normal((SIZE,SIZE,3),dtype=np.float32) for i in ids])
        def gauss(mask,sigma):return np.round(np.clip(marked.astype(np.float32)/255+sigma*noise*mask,0,1)*255).astype(np.uint8)
        for sigma in SIGMAS:conditions[f'global-gaussian-{sigma:.2f}']=gauss(1.,sigma)
        conditions['global-energy-matched']=gauss(1.,.05*math.sqrt(2458/16384))
        for area,k in enumerate(AREAS):
            for r,region in enumerate(('object','background')):
                mask=mask_array(bank,np.arange(len(ids)),r,area)[...,None]
                conditions[f'erase-{region}-{k}']=np.where(mask,cover,marked)
                for sigma in SIGMAS:conditions[f'gaussian-{region}-{k}-{sigma:.2f}']=gauss(mask,sigma)
        for size in (256,512):conditions[f'resize-{size}']=np.stack([np.array(Image.fromarray(x).resize((size,size),Image.Resampling.BILINEAR).resize((SIZE,SIZE),Image.Resampling.BILINEAR)) for x in marked])
        for size in (256,512):
            for r,region in enumerate(('object','background')):
                mm=mask_array(bank,np.arange(len(ids)),r)[...,None]
                conditions[f'local-resize-{size}-{region}']=np.where(mm,conditions[f'resize-{size}'],marked)
        for name,images in conditions.items():
            p=RUN/arm/name/'0000.npz'
            if p.exists():assert str(np.load(p)['identity'])==identity;continue
            deadline();z=decode(model,images);energy=(((images.astype(np.float32)-marked)/255.)**2).mean((1,2,3));save_condition(arm,name,ids,bits,clean,z,identity,energy=energy)
        # Unmarked negatives and wrong-message controls use same decoder.
        save_condition(arm,'unmarked',ids,bits,clean,decode(model,cover),identity)
        stamp('cheap',completed_arm=arm);print('cheap complete',arm,flush=True)
        del model,conditions,noise;gc.collect()
    write(RUN/'cheap-completion.json',dict(identity=identity,models=len(h['arms']),images=len(ids)))

def load_editor(name):
    if name=='lama':
        assert sha(LAMA_PATH)==LAMA_SHA;model=torch.jit.load(str(LAMA_PATH),map_location=DEVICE).eval();return model,dict(sha256=LAMA_SHA)
    assert StableDiffusionInpaintPipeline is not None
    path=EDITOR_ROOT/name;lock=json.loads((path/'editor-lock.json').read_text());assert lock['revision']==EDITORS[name]['revision'] and lock['model_id']==EDITORS[name]['model_id']
    for fn,digest in lock['files'].items():assert sha(path/fn)==digest
    pipe=StableDiffusionInpaintPipeline.from_pretrained(str(path),torch_dtype=torch.float16,variant='fp16',use_safetensors=True,local_files_only=True)
    pipe.scheduler=DDIMScheduler.from_config(pipe.scheduler.config);pipe.to(DEVICE);pipe.set_progress_bar_config(disable=True)
    return pipe,dict(lock_sha=sha(path/'editor-lock.json'),scheduler=dict(pipe.scheduler.config))

@torch.inference_mode()
def edit(editor,name,rgb,masks,ids,region):
    if name=='lama':
        large=np.stack([np.array(Image.fromarray(a).resize((256,256),Image.Resampling.BILINEAR)) for a in rgb])
        x=tensor(large)
        m=F.interpolate(torch.from_numpy(masks.copy()).to(DEVICE,dtype=torch.float32)[:,None],size=(256,256),mode='nearest')
        raw=editor(x,m)
        if isinstance(raw,dict):raw=raw['inpainted']
        large_rgb=to_rgb(raw)
        small=np.stack([np.array(Image.fromarray(a).resize((SIZE,SIZE),Image.Resampling.BILINEAR)) for a in large_rgb]);filtered=np.zeros(len(rgb),bool)
    else:
        ims=[Image.fromarray(a).resize((512,512),Image.Resampling.BILINEAR) for a in rgb]
        mm=[Image.fromarray(a.astype(np.uint8)*255).resize((512,512),Image.Resampling.NEAREST) for a in masks]
        outputs=editor(prompt=[PROMPT]*len(rgb),negative_prompt=[NEGATIVE]*len(rgb),image=ims,mask_image=mm,height=512,width=512,num_inference_steps=STEPS,strength=STRENGTH,guidance_scale=GUIDANCE,eta=0.,generator=[torch.Generator('cpu').manual_seed(seed_for(i,region)) for i in ids])
        small=np.stack([np.array(im.resize((SIZE,SIZE),Image.Resampling.BILINEAR)) for im in outputs.images])
        filtered=np.array(outputs.nsfw_content_detected if outputs.nsfw_content_detected is not None else [False]*len(rgb),dtype=bool)
    composed=np.where(masks[...,None],small,rgb)
    assert np.array_equal(composed[~masks],rgb[~masks])
    change=np.mean(np.abs(composed.astype(np.float32)-rgb),axis=3)
    change=(change*masks).sum((1,2))/masks.sum((1,2))/255.
    return composed,filtered,change

def editor_evaluation(name,cover,bank,bits,ids,h,identity):
    editor,lock=load_editor(name);write(RUN/(name+'-lock.json'),lock)
    bs=LAMA_BATCH if name=='lama' else EDITOR_BATCH
    for r,region in enumerate(('object','background')):
        masks=mask_array(bank,np.arange(len(ids)),r)
        # Shared original-fed synthesis is made exactly once per image and region.
        originals=[];flags=[]
        for lo in range(0,len(ids),bs):
            deadline();idx=np.arange(lo,min(lo+bs,len(ids)));p=RUN/'shared'/f'{name}-{region}'/f'{lo:04d}.npz'
            if p.exists():
                d=np.load(p);assert str(d['identity'])==identity;out=d['images'];flag=d['filtered']
            else:
                out,flag,change=edit(editor,name,cover[idx],masks[idx],ids[idx],r);pack(p,images=out,filtered=flag,ids=ids[idx],identity=np.array(identity),change_mae=change)
            originals.append(out);flags.append(flag)
        fill=np.concatenate(originals);fill_flags=np.concatenate(flags)
        for arm,info in h['arms'].items():
            model,marked,clean=ensure_clean(arm,info,cover,bits,ids,identity)
            for lo in range(0,len(ids),bs):
                deadline();idx=np.arange(lo,min(lo+bs,len(ids)))
                for mode in ('B','C'):
                    cond=f'{name}-{mode}-{region}';p=RUN/arm/cond/f'{lo:04d}.npz'
                    if p.exists():assert str(np.load(p)['identity'])==identity;continue
                    t=time.perf_counter()
                    if mode=='B':
                        out=np.where(masks[idx,...,None],fill[idx],marked[idx]);flag=fill_flags[idx]
                        change=(np.abs(out.astype(np.float32)-marked[idx]).mean(3)*masks[idx]).sum((1,2))/2458/255.
                    else:out,flag,change=edit(editor,name,marked[idx],masks[idx],ids[idx],r)
                    assert np.array_equal(out[~masks[idx]],marked[idx][~masks[idx]])
                    z=decode(model,out);save_condition(arm,cond,ids[idx],bits[idx],clean[idx],z,identity,flag,out if mode=='C' else None,change,lo)
                    sync();stamp('evaluation',editor=name,region=region,arm=arm,mode=mode,completed_images=int(idx[-1])+1,seconds=time.perf_counter()-t)
                if lo%128==0:print(name,region,arm,int(idx[-1])+1,'/',len(ids),flush=True)
            del model;gc.collect()
    del editor;gc.collect()
    if CUDA:torch.cuda.empty_cache()
    write(RUN/(name+'-completion.json'),dict(identity=identity,models=len(h['arms']),images=len(ids),marked_edits=20*2*len(ids),shared_edits=2*len(ids)))

def technical(cover,bank,bits,ids):
    # Untrained technical marking is permitted only on validation; no scientific result.
    model=new_model().eval();info=dict(alpha=1.);marked=marking(model,info,cover,bits);masks=mask_array(bank,np.arange(len(ids)),0)
    records=[]
    for name in BENCH_EDITORS:
        editor,lock=load_editor(name);candidates=BENCH_BATCHES if name!='lama' else (16,32)
        for bs in candidates:
            if PROFILE=='smoke':bs=2
            try:
                edit(editor,name,marked[:2],masks[:2],ids[:2],0);sync();t=time.perf_counter()
                out,flags,changes=edit(editor,name,marked[:bs],masks[:bs],ids[:bs],0);sync();elapsed=time.perf_counter()-t
                assert np.isfinite(decode(model,out)).all();assert np.array_equal(out[~masks[:bs]],marked[:bs][~masks[:bs]])
                records.append(dict(editor=name,batch=bs,seconds=elapsed,seconds_per_image=elapsed/bs,projected_21000_seconds=21000*elapsed/bs,filtered=int(flags.sum()),minimum_change=float(changes.min())))
                print(records[-1],flush=True);write(RUN/'benchmark.json',records)
            except torch.cuda.OutOfMemoryError:
                records.append(dict(editor=name,batch=bs,status='oom'));torch.cuda.empty_cache();break
            if PROFILE=='smoke':break
        del editor;gc.collect();torch.cuda.empty_cache()
    stamp('technical_complete',scientific_result=False)


In [ ]:
def report():
    perceptual=lpips.LPIPS(net='alex').to(DEVICE).eval().requires_grad_(False)
    assert sha(DATA/'test.npy')==json.loads((DATA/'identity.json').read_text())['files']['test.npy']
    covers=np.load(DATA/'test.npy')
    lpips_weights=list(Path(os.environ['TORCH_HOME']).rglob('*.pth'))+[Path(lpips.__file__).parent/'weights/v0.1/alex.pth']
    write(RUN/'lpips-weights.json',{str(p):sha(p) for p in lpips_weights})
    h=json.loads((RUN/'frozen-handoff.json').read_text());arms=list(h['arms']);N=500
    # Recompute from raw logits; never average unpaired region subsets silently.
    data={};clean={};quality={};summary=[];per_bit=[];reference_ids=None;reference_bits=None
    for arm in arms:
        d=np.load(RUN/arm/'clean.npz');bits=d['bits'];ids=d['ids'];assert len(ids)==N
        if reference_ids is None:reference_ids=ids.copy();reference_bits=bits.copy()
        else:assert np.array_equal(ids,reference_ids) and np.array_equal(bits,reference_bits)
        ce=(d['logits']>=0)!=bits;clean[arm]=ce
        distances=[]
        with torch.inference_mode():
            for lo in range(0,N,32):
                value=perceptual(tensor(covers[lo:lo+32])*2-1,tensor(d['marked'][lo:lo+32])*2-1).flatten().cpu().numpy();distances.extend(value.tolist())
        pack(RUN/arm/'lpips.npz',ids=ids,lpips=np.array(distances))
        quality[arm]=dict(psnr=-10*float(np.log10(d['mse'].mean())),ssim=float(d['ssim'].mean()),lpips=float(np.mean(distances)))
        data[arm,'clean']=(ce,np.zeros(N,bool))
        for folder in (RUN/arm).iterdir():
            if not folder.is_dir():continue
            chunks=[np.load(p) for p in sorted(folder.glob('*.npz'))]
            if not chunks:continue
            ci=np.concatenate([c['ids'] for c in chunks]);order=np.argsort(ci);expected=np.argsort(ids)
            if len(ci)!=N or len(set(ci.tolist()))!=N:continue
            assert np.array_equal(ci[order],ids[expected])
            lookup={int(i):j for j,i in enumerate(ci)};perm=np.array([lookup[int(i)] for i in ids])
            zz=np.concatenate([c['logits'] for c in chunks])[perm];bb=np.concatenate([c['bits'] for c in chunks])[perm];assert np.array_equal(bb,bits)
            ff=np.concatenate([c['filtered'] for c in chunks])[perm]
            data[arm,folder.name]=((zz>=0)!=bits,ff)
    conditions=sorted(set(c for a,c in data))
    for cond in conditions:
        present=[a for a in arms if (a,cond) in data]
        if len(present)!=len(arms):continue
        common=~np.logical_or.reduce([data[a,cond][1] for a in arms])
        for arm in arms:
            err,ff=data[arm,cond];ce=clean[arm];e=err[common];c=ce[common]
            per_bit.extend(dict(arm=arm,condition=cond,bit=j,n=int(common.sum()),errors=int(e[:,j].sum()),new_errors=int((e[:,j]&~c[:,j]).sum())) for j in range(BITS))
            summary.append(dict(arm=arm,condition=cond,n=int(common.sum()),filtered=int(ff.sum()),ber_pct=float(e.mean()*100) if len(e) else None,delta_ber_pp=float((e.mean()-c.mean())*100) if len(e) else None,exact_pct=float((~e.any(1)).mean()*100) if len(e) else None,new_errors=int((e&~c).sum()),corrected_errors=int((~e&c).sum()),initial_correct_bits=int((~c).sum()),psnr=quality[arm]['psnr'],ssim=quality[arm]['ssim'],lpips=quality[arm]['lpips']))
    csvwrite(RUN/'summary.csv',summary)
    csvwrite(RUN/'per-bit.csv',per_bit)
    rng=np.random.default_rng(BOOT_SEED);draws=rng.integers(0,N,(BOOT_N,N));contrasts=[]
    def interval(values,valid):
        a=np.where(valid,values,np.nan);boot=np.nanmean(a[draws],axis=1);return float(np.nanmean(a)),np.nanquantile(boot,[.025,.975]).tolist()
    def delta(arm,cond):return data[arm,cond][0].mean(1)-clean[arm].mean(1)
    def add(label,family,region,values,valid,primary=True):
        mean,ci=interval(values*100,valid)
        # Paired sign-flip randomization, centered null; image is exchangeability unit.
        v=values[valid]*100;rg=np.random.default_rng(BOOT_SEED+1);null=[]
        for _ in range(100):null.extend((rg.choice([-1.,1.],size=(100,len(v)))*v).mean(1).tolist())
        p=(1+np.count_nonzero(np.abs(null)>=abs(mean)))/(1+len(null))
        contrasts.append(dict(contrast=label,family=family,region=region,n=int(valid.sum()),effect_pp=mean,ci_low_pp=ci[0],ci_high_pp=ci[1],p_randomization=float(p),primary=primary))
    for family in ('gaussian','erase','mix'):
        a,b=family+'-obj010',family+'-obj090';balanced=family+'-obj050'
        for r,region in enumerate(('object','background')):
            er='erase-'+region+'-2458'
            for cond in [er]+[f'{ed}-C-{region}' for ed in ('lama','sd15','sd2')]:
                if (a,cond) not in data or (b,cond) not in data:continue
                valid=~(data[a,cond][1]|data[b,cond][1]);sign=1 if r==0 else -1
                add('spatial:'+cond,family,region,sign*(delta(a,cond)-delta(b,cond)),valid)
                if cond!=er and (a,er) in data and (b,er) in data:add('attenuation:A-'+cond,family,region,sign*((delta(a,er)-delta(b,er))-(delta(a,cond)-delta(b,cond))),valid)
            for ed in ('lama','sd15','sd2'):
                B=f'{ed}-B-{region}';C=f'{ed}-C-{region}'
                if all((arm,c) in data for arm in (a,b) for c in (B,C)):
                    valid=~np.logical_or.reduce([data[arm,c][1] for arm in (a,b) for c in (B,C)])
                    add('diagnostic:B-C:'+ed,family,region,(1 if r==0 else -1)*((delta(a,B)-delta(b,B))-(delta(a,C)-delta(b,C))),valid,False)
        for ed in ('lama','sd15','sd2'):
            cc=[f'{ed}-C-{r}' for r in ('object','background')]
            if not all((arm,c) in data for arm in (a,b,balanced) for c in cc):continue
            valid=~np.logical_or.reduce([data[arm,c][1] for arm in (a,b,balanced) for c in cc])
            for other in (a,b):
                # Worst-region is max of regional means, recomputed within each bootstrap.
                vv={arm:np.stack([data[arm,c][0].mean(1) for c in cc]) for arm in (other,balanced)}
                mean=(vv[other][:,valid].mean(1).max()-vv[balanced][:,valid].mean(1).max())*100
                bootstrap=[]
                for indices in draws:
                    take=indices[valid[indices]]
                    bootstrap.append((vv[other][:,take].mean(1).max()-vv[balanced][:,take].mean(1).max())*100)
                ci=np.quantile(bootstrap,[.025,.975]);contrasts.append(dict(contrast='balanced-worst:'+ed+':'+other,family=family,region='worst',n=int(valid.sum()),effect_pp=float(mean),ci_low_pp=float(ci[0]),ci_high_pp=float(ci[1]),p_randomization=None,primary=True))
    # Holm correction for the finite primary randomization-test family; worst-region CIs descriptive.
    prim=[i for i,x in enumerate(contrasts) if x['primary'] and x['p_randomization'] is not None];rank=sorted(prim,key=lambda i:contrasts[i]['p_randomization']);running=0.
    for j,i in enumerate(rank):running=max(running,min(1.,(len(rank)-j)*contrasts[i]['p_randomization']));contrasts[i]['p_holm']=running
    for x in contrasts:x.setdefault('p_holm',None)
    csvwrite(RUN/'contrasts.csv',contrasts)
    # Common clean-correct paired message survival for the principal 10/90 vs90/10 comparison.
    survival=[]
    for family in ('gaussian','erase','mix'):
        a,b=family+'-obj010',family+'-obj090'
        for ed in ('lama','sd15','sd2'):
            for region in ('object','background'):
                c=f'{ed}-C-{region}'
                if not all((arm,c) in data for arm in (a,b)):continue
                valid=~(clean[a].any(1)|clean[b].any(1)|data[a,c][1]|data[b,c][1]);n=int(valid.sum())
                survival.append(dict(family=family,editor=ed,region=region,n=n,exact_10=int((~data[a,c][0][valid].any(1)).sum()),exact_90=int((~data[b,c][0][valid].any(1)).sum())))
    if survival:csvwrite(RUN/'common-clean-survival.csv',survival)
    write(RUN/'report-status.json',dict(models=len(arms),images=N,conditions=conditions,completed_panels=[s for s in ('cheap','lama','sd15','sd2') if (RUN/(s+'-completion.json')).exists()],single_seed=True,ci_conditional_on_weights=True,quality=quality,primary_p_adjustment='Holm over primary spatial and attenuation randomization tests; worst-region intervals descriptive'))
    # Compact heatmap of absolute BER for main channels.
    cols=['clean','erase-object-2458','erase-background-2458']+[f'{e}-C-{r}' for e in ('lama','sd15','sd2') for r in ('object','background')]
    vals=np.full((len(arms),len(cols)),np.nan)
    for i,a in enumerate(arms):
        for j,c in enumerate(cols):
            rr=next((r for r in summary if r['arm']==a and r['condition']==c),None)
            if rr and rr['ber_pct'] is not None:vals[i,j]=rr['ber_pct']
    fig,ax=plt.subplots(figsize=(14,10));im=ax.imshow(vals,aspect='auto',cmap='magma_r');ax.set_yticks(range(len(arms)),arms);ax.set_xticks(range(len(cols)),cols,rotation=40,ha='right');fig.colorbar(im,ax=ax,label='BER (%)');fig.tight_layout();fig.savefig(RUN/'ber-heatmap.png',dpi=170);plt.close(fig)
    stamp('reported',complete_panels=[s for s in ('cheap','lama','sd15','sd2') if (RUN/(s+'-completion.json')).exists()])


In [ ]:
if PROFILE=='report':report()
else:
    cover,bank,bits,ids=load_bank()
    if PROFILE in ('smoke','benchmark'):technical(cover,bank,bits,ids)
    else:
        h=load_handoff();frozen=RUN/'frozen-handoff.json'
        if frozen.exists():assert json.loads(frozen.read_text())==h
        else:write(frozen,h)
        identity=fingerprint(dict(source=SOURCE_SHA,config=CONFIG,handoff_sha=sha(TRAIN_RUN/'handoff.json'),data_sha=sha(DATA/'identity.json'),torch=str(torch.__version__)))
        if PROFILE=='cheap':cheap(cover,bank,bits,ids,h,identity)
        else:editor_evaluation(PROFILE,cover,bank,bits,ids,h,identity)
